# MILK10k — Fine-tuning (DINOv2-base)

`01_baseline` notebook'unda görüntü modelini hiç eğitmedik: donmuş DINOv2-large özellikleri + Logistic Regression. Bu notebook'ta görüntü modelinin kendisini (DINOv2-base) bizim lezyonlarla eğitiyoruz.

| | 01_baseline (donmuş) | 02_finetune |
|---|---|---|
| Eğitilen kısım | sadece LR (~45 bin parametre) | DINOv2-base'in tamamı + yeni sınıflandırma katmanı (~86 milyon) |
| İki görüntü | hazır vektörler yan yana | aynı model iki görüntüye bakıyor, birlikte eğitiliyor |
| Augmentation | yok | rastgele kırpma, döndürme, aynalama, renk oynatma |

Son sistem üç modelin olasılık ortalaması: **fine-tuned DINOv2-base + donmuş DINOv2-large + tabular LightGBM** (1/3'er). Bu birleşim fold 0 pilotunda seçilip sabitlendi (pilot fold 0: 0.547; aynı fold'da 01'in sistemi 0.517).

**Çalıştırma:** Run All. İlerleme çubukları yüzde ve kalan süreyi gösterir.
- Süre (RTX 4050): fold başına ~25–35 dk, son eğitim ~30–40 dk.
- Biten fold'lar `outputs/cache/finetune/` altına kaydedilir. Notebook yarıda kesilirse tekrar çalıştırınca biten fold'lar atlanır.
- Önce `01_baseline.ipynb` çalışmış olmalı: donmuş DINOv2-large özellikleri onun cache'inden okunuyor.
- Eğitim sırasında GPU kullanan oyun vb. açık olmasın; video izlemek sorun değil.

In [1]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoModel

pd.set_option("display.max_columns", 50)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
DATA = ROOT / "data"
CACHE = ROOT / "outputs" / "cache"
FT_DIR = CACHE / "finetune"            # fold sonuçları ve test tahminleri
OUT = ROOT / "outputs" / "finetune"    # submission ve sonuç tabloları
FT_DIR.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT / "notebooks"))
from ft_data import LesionDataset, preprocess  # veri yükleme: notebooks/ft_data.py

CLASSES = ["AKIEC", "BCC", "BEN_OTH", "BKL", "DF", "INF", "MAL_OTH", "MEL", "NV", "SCCKA", "VASC"]
IMAGE_TYPES = {"clinical: close-up": "clin", "dermoscopic": "derm"}
N_FOLDS = 5
SEED = 42

# fine-tuning ayarları
EPOCHS = 10
BATCH = 16             # lezyon (= 32 görüntü)
LR_BACKBONE = 1e-5     # önceden öğrenilenleri bozmamak için çok küçük
LR_HEAD = 1e-3         # yeni sınıflandırma katmanı sıfırdan öğreniyor
NUM_WORKERS = 4
RUN_FINAL = True       # CV'den sonra tüm train ile son eğitim + test submission
MAX_STEPS = None       # hızlı deneme için epoch başına adım sınırı (None = tam epoch)

## 1. Veri ve fold'lar

`01_baseline` ile aynı: lezyon başına tek satır, aynı tabular özellikler ve aynı 5 fold (seed 42). Böylece sonuçlar 01 ile doğrudan kıyaslanabiliyor.

In [2]:
def load_split(split):
    meta = pd.read_csv(DATA / f"MILK10k_{split}_Metadata.csv")
    meta["kind"] = meta["image_type"].map(IMAGE_TYPES)
    meta["path"] = [str(DATA / f"MILK10k_{split}_Input" / l / f"{i}.jpg") for l, i in zip(meta["lesion_id"], meta["isic_id"])]
    monet_cols = [c for c in meta.columns if c.startswith("MONET_")]

    patient = meta.drop_duplicates("lesion_id").set_index("lesion_id")[["age_approx", "sex", "skin_tone_class", "site"]]
    monet = meta.pivot(index="lesion_id", columns="kind", values=monet_cols)
    monet.columns = [f"{kind}_{col}" for col, kind in monet.columns]
    paths = meta.pivot(index="lesion_id", columns="kind", values="path")
    return patient.join(monet).join(paths)


def tabular_features(df, sites):
    X = df.drop(columns=list(IMAGE_TYPES.values()))
    X["sex"] = pd.Categorical(X["sex"], categories=["female", "male"])
    X["site"] = pd.Categorical(X["site"], categories=sites)
    return X


train, test = load_split("Training"), load_split("Test")
gt = pd.read_csv(DATA / "MILK10k_Training_GroundTruth.csv").set_index("lesion_id").loc[train.index]
assert list(gt.columns) == CLASSES
y = gt.values.argmax(1)

sites = sorted(train["site"].dropna().unique())
X_tab, X_tab_test = tabular_features(train, sites), tabular_features(test, sites)
folds = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(X_tab, y))
print(f"train: {len(train)} lezyon | test: {len(test)} lezyon | fold başına ~{len(folds[0][1])} doğrulama lezyonu")

train: 5240 lezyon | test: 479 lezyon | fold başına ~1048 doğrulama lezyonu


## 2. Referans modeller (01'den)

Fine-tuned modelin yanına konacak iki model, `01_baseline`'daki ayarlarla:
- **Tabular:** LightGBM
- **Donmuş görüntü modeli:** DINOv2-large özellikleri + Logistic Regression (C=0.03)

İkisinin de out-of-fold olasılıkları burada hesaplanıyor (~1–2 dk).

In [3]:
def tabular_model():
    return LGBMClassifier(n_estimators=300, learning_rate=0.03, num_leaves=15, min_child_samples=10,
                          subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                          class_weight="balanced", random_state=SEED, verbose=-1)


def frozen_model():
    return make_pipeline(StandardScaler(), LogisticRegression(C=0.03, class_weight="balanced", max_iter=3000))


def take(X, idx):
    return X.iloc[idx] if isinstance(X, pd.DataFrame) else X[idx]


def oof_proba(make_model, X):
    proba = np.zeros((len(y), len(CLASSES)))
    for tr, va in folds:
        proba[va] = make_model().fit(take(X, tr), y[tr]).predict_proba(take(X, va))
    return proba


def dinov2_large_features(df, split):
    cache = CACHE / f"dinov2_large_{split}.npz"
    assert cache.exists(), "önce 01_baseline.ipynb'yi BACKBONE = 'dinov2_large' ile çalıştır"
    z = np.load(cache)
    assert (z["lesion_id"] == df.index.values.astype(str)).all()
    return z["X"]


def f1(idx, p):
    return f1_score(y[idx], p.argmax(1), average="macro")


X_frozen, X_frozen_test = dinov2_large_features(train, "train"), dinov2_large_features(test, "test")
p_tab = oof_proba(tabular_model, X_tab)
p_frozen = oof_proba(frozen_model, X_frozen)
all_idx = np.arange(len(y))
print(f"tabular {f1(all_idx, p_tab):.4f} | donmuş DINOv2-large {f1(all_idx, p_frozen):.4f} | "
      f"01'in sistemi (ikisinin ortalaması) {f1(all_idx, (p_frozen + p_tab) / 2):.4f}")

tabular 0.3334 | donmuş DINOv2-large 0.4837 | 01'in sistemi (ikisinin ortalaması) 0.4869


## 3. Fine-tuning

**Model:** Klinik ve dermoskopik görüntü **aynı** DINOv2-base'ten geçiyor. Her görüntüden CLS token + patch token ortalaması alınıp iki görüntünün özellikleri yan yana konuyor, oradan 11 sınıfa giden tek bir doğrusal katman var.

**Eğitim:**
- AdamW; backbone için çok küçük öğrenme hızı (1e-5), yeni katman için 1e-3.
- İlk epoch'ta yavaş ısınma (warmup), sonra kosinüs şeklinde azalan öğrenme hızı.
- Sınıf ağırlıklı (balanced) cross-entropy; mixed precision (fp16); gradient clipping.
- **Gradient checkpointing** sadece GPU'da yer azsa açılıyor: hafızayı ~2 GB'a indirir ama ~%20–25 yavaşlatır.
- Her fold'da **son epoch** kullanılıyor. En iyi epoch'u doğrulama fold'una bakarak seçmek skoru şişirirdi.

In [4]:
class TwoViewDino(nn.Module):
    """Klinik ve dermoskopik görüntü aynı DINOv2'den geçer; iki görüntünün özellikleri yan yana -> sınıflandırma katmanı."""

    def __init__(self, ckpt):
        super().__init__()
        self.backbone = AutoModel.from_pretrained("facebook/dinov2-base")
        if ckpt:
            self.backbone.gradient_checkpointing_enable()
        d = self.backbone.config.hidden_size
        self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(4 * d, len(CLASSES)))

    def embed(self, x):
        out = self.backbone(pixel_values=x)
        return torch.cat([out.pooler_output, out.last_hidden_state[:, 1:].mean(1)], 1)  # CLS + patch ortalaması

    def forward(self, clin, derm):
        e = self.embed(torch.cat([clin, derm]))
        return self.head(torch.cat([e[:len(clin)], e[len(clin):]], 1))


def make_loader(df, idx, labels, train_mode):
    ds = LesionDataset(df["clin"].values[idx], df["derm"].values[idx], labels, train_mode)
    return DataLoader(ds, batch_size=BATCH if train_mode else 32, shuffle=train_mode, drop_last=train_mode,
                      num_workers=NUM_WORKERS, persistent_workers=train_mode, pin_memory=True)


def steps_per_epoch(n_lesions):
    n = n_lesions // BATCH
    return n if MAX_STEPS is None else min(MAX_STEPS, n)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    probs = []
    for clin, derm, _ in loader:
        with torch.autocast("cuda", dtype=torch.float16):
            probs.append(torch.softmax(model(clin.cuda(), derm.cuda()).float(), 1).cpu().numpy())
    return np.concatenate(probs)


def train_model(tr_idx, progress, on_epoch=None):
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    ckpt = torch.cuda.mem_get_info()[0] / 2**30 < 5.3
    loader = make_loader(train, tr_idx, y[tr_idx], train_mode=True)
    model = TwoViewDino(ckpt).cuda()
    opt = torch.optim.AdamW([{"params": model.backbone.parameters(), "lr": LR_BACKBONE},
                             {"params": model.head.parameters(), "lr": LR_HEAD}], weight_decay=0.05)
    n_steps = steps_per_epoch(len(tr_idx))
    total = EPOCHS * n_steps
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / n_steps) * 0.5 * (1 + np.cos(np.pi * min(1.0, s / total))))
    class_w = len(tr_idx) / (len(CLASSES) * np.bincount(y[tr_idx], minlength=len(CLASSES)))
    loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(class_w, dtype=torch.float32, device="cuda"))
    scaler = torch.amp.GradScaler()
    progress.write(f"gradient checkpointing: {'açık' if ckpt else 'kapalı'}")

    for epoch in range(EPOCHS):
        model.train()
        t, losses = time.time(), []
        for step, (clin, derm, labels) in enumerate(loader):
            if step == n_steps:
                break
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(clin.cuda(non_blocking=True), derm.cuda(non_blocking=True))
            loss = loss_fn(logits.float(), labels.cuda())
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            losses.append(loss.item())
            progress.update(1)
        if on_epoch:
            on_epoch(model, epoch, float(np.mean(losses)), time.time() - t)
    return model

### 5-fold eğitim

Her fold için model kalan 4 fold'la eğitilip o fold tahmin ediliyor. Her epoch sonunda doğrulama skorları yazılıyor:
- **fine-tuned:** modelin tek başına skoru
- **üçlü:** ana sistem, yani fine-tuned + donmuş + tabular ortalaması

Daha önce bitmiş fold'lar (`outputs/cache/finetune/fold*.npz`) atlanıyor.

In [5]:
preprocess(list(train["clin"]) + list(train["derm"]) + list(test["clin"]) + list(test["derm"]))


def fold_file(k):
    return FT_DIR / f"fold{k}.npz"


todo = [k for k in range(N_FOLDS) if not fold_file(k).exists()]
print("hazır fold'lar:", [k for k in range(N_FOLDS) if k not in todo], "| eğitilecek:", todo)
progress = tqdm(total=sum(EPOCHS * steps_per_epoch(len(folds[k][0])) for k in todo), desc="CV fine-tuning", unit="adım")

for k in todo:
    tr, va = folds[k]
    va_loader = make_loader(train, va, y[va], train_mode=False)
    last = {}

    def log(model, epoch, loss, sec):
        last["p"] = p = predict(model, va_loader)
        progress.write(f"fold {k} | epoch {epoch + 1:2d}/{EPOCHS} | loss {loss:.3f} | val: fine-tuned {f1(va, p):.4f}, "
                       f"üçlü {f1(va, (p + p_frozen[va] + p_tab[va]) / 3):.4f} | {sec / 60:.1f} dk")

    model = train_model(tr, progress, on_epoch=log)
    np.savez(fold_file(k), va=va, p_va=last["p"])
    del model
    torch.cuda.empty_cache()
progress.close()

küçültme: 0it [00:00, ?it/s]

hazır fold'lar: [0, 1, 2] | eğitilecek: [3, 4]


CV fine-tuning:   0%|          | 0/5240 [00:00<?, ?adım/s]

gradient checkpointing: açık
fold 3 | epoch  1/10 | loss 2.184 | val: fine-tuned 0.3633, üçlü 0.4831 | 2.8 dk
fold 3 | epoch  2/10 | loss 1.738 | val: fine-tuned 0.4212, üçlü 0.4978 | 2.4 dk
fold 3 | epoch  3/10 | loss 1.359 | val: fine-tuned 0.4274, üçlü 0.4851 | 2.4 dk
fold 3 | epoch  4/10 | loss 0.977 | val: fine-tuned 0.3913, üçlü 0.5204 | 2.4 dk
fold 3 | epoch  5/10 | loss 0.775 | val: fine-tuned 0.4999, üçlü 0.5074 | 2.4 dk
fold 3 | epoch  6/10 | loss 0.541 | val: fine-tuned 0.4779, üçlü 0.5111 | 3.0 dk
fold 3 | epoch  7/10 | loss 0.415 | val: fine-tuned 0.4812, üçlü 0.5177 | 3.3 dk
fold 3 | epoch  8/10 | loss 0.279 | val: fine-tuned 0.5064, üçlü 0.5282 | 3.3 dk
fold 3 | epoch  9/10 | loss 0.242 | val: fine-tuned 0.5157, üçlü 0.5365 | 3.3 dk
fold 3 | epoch 10/10 | loss 0.172 | val: fine-tuned 0.5157, üçlü 0.5378 | 2.8 dk
gradient checkpointing: açık
fold 4 | epoch  1/10 | loss 2.254 | val: fine-tuned 0.3223, üçlü 0.4600 | 2.9 dk
fold 4 | epoch  2/10 | loss 1.793 | val: fine-tuned

## 4. Sonuçlar (5-fold CV)

Bütün sistemler aynı fold'larda, `01_baseline`'ın sistemiyle (donmuş + tabular) kıyaslanıyor. Karar **ana sisteme** (fine-tuned + donmuş + tabular) göre verilecek; diğer satırlar bilgi için.

In [6]:
p_ft = np.zeros_like(p_tab)
for k in range(N_FOLDS):
    z = np.load(fold_file(k))
    assert (z["va"] == folds[k][1]).all(), f"fold {k} dosyası farklı bir bölünmeye ait"
    p_ft[z["va"]] = z["p_va"]

systems = {
    "01: donmuş + tabular": (p_frozen + p_tab) / 2,
    "fine-tuned": p_ft,
    "fine-tuned + tabular": (p_ft + p_tab) / 2,
    "fine-tuned + donmuş": (p_ft + p_frozen) / 2,
    "fine-tuned + donmuş + tabular (ana sistem)": (p_ft + p_frozen + p_tab) / 3,
}


def fold_scores(p):
    return np.array([f1(va, p[va]) for _, va in folds])


ref = fold_scores(systems["01: donmuş + tabular"])
results = pd.DataFrame({name: {"macro_f1": f1(all_idx, p), "fold_std": fold_scores(p).std(),
                               "01'e göre": fold_scores(p).mean() - ref.mean(),
                               "01'i geçtiği fold": int((fold_scores(p) > ref).sum())}
                        for name, p in systems.items()}).T
results.to_csv(OUT / "cv_results.csv")
results.round(4)

,macro_f1,fold_std,01'e göre,01'i geçtiği fold
01: donmuş + tabular,0.4869,0.0242,0.0000,0.0
fine-tuned,0.5265,0.0146,0.0375,5.0
fine-tuned + tabular,0.5201,0.0168,0.0319,5.0
fine-tuned + donmuş,0.5445,0.0128,0.0551,5.0
fine-tuned + donmuş + tabular (ana sistem),0.5441,0.0062,0.0525,5.0


In [7]:
per_class = pd.DataFrame({name: f1_score(y, p.argmax(1), average=None) for name, p in systems.items()}, index=CLASSES).round(3)
per_class.insert(0, "n", np.bincount(y))
per_class.to_csv(OUT / "cv_per_class_f1.csv")
per_class

,n,01: donmuş + tabular,fine-tuned,fine-tuned + tabular,fine-tuned + donmuş,fine-tuned + donmuş + tabular (ana sistem)
AKIEC,303,0.424,0.537,0.534,0.542,0.540
BCC,2522,0.859,0.883,0.888,0.895,0.891
BEN_OTH,44,0.254,0.200,0.194,0.194,0.211
BKL,544,0.450,0.544,0.537,0.563,0.566
DF,52,0.506,0.554,0.520,0.566,0.596
INF,50,0.152,0.196,0.184,0.230,0.225
MAL_OTH,9,0.000,0.000,0.000,0.000,0.000
MEL,450,0.595,0.609,0.639,0.642,0.637
NV,746,0.775,0.781,0.800,0.792,0.804
SCCKA,473,0.651,0.705,0.707,0.733,0.718


## 5. Son eğitim ve test submission

Fine-tuned model aynı ayarlarla **tüm** train verisiyle (5240 lezyon) eğitilip test seti tahmin ediliyor. Donmuş model ve tabular model de tüm train verisiyle eğitiliyor. Üçünün ortalamasında en olası sınıfa 1, diğerlerine 0 yazılıyor.

In [8]:
if RUN_FINAL:
    test_file = FT_DIR / "test.npz"
    if test_file.exists():
        p_ft_test = np.load(test_file)["p_test"]
    else:
        progress = tqdm(total=EPOCHS * steps_per_epoch(len(all_idx)), desc="son eğitim (tüm train)", unit="adım")
        model = train_model(all_idx, progress, on_epoch=lambda m, epoch, loss, sec: progress.write(
            f"epoch {epoch + 1:2d}/{EPOCHS} | loss {loss:.3f} | {sec / 60:.1f} dk"))
        progress.close()
        te_loader = make_loader(test, np.arange(len(test)), np.zeros(len(test), int), train_mode=False)
        p_ft_test = predict(model, te_loader)
        np.savez(test_file, p_test=p_ft_test, lesion_id=test.index.values.astype(str))
        del model
        torch.cuda.empty_cache()

    p_frozen_test = frozen_model().fit(X_frozen, y).predict_proba(X_frozen_test)
    p_tab_test = tabular_model().fit(X_tab, y).predict_proba(X_tab_test)
    p_test = (p_ft_test + p_frozen_test + p_tab_test) / 3

    # ilk sütun 'lesion_id' olmalı: yarışma sayfası 'lesion' diyor ama skorlama kodu 'lesion_id' istiyor
    sub = pd.DataFrame(np.eye(len(CLASSES))[p_test.argmax(1)], columns=CLASSES, index=test.index).rename_axis("lesion_id")
    assert len(sub) == len(test) and (sub.sum(axis=1) == 1).all()
    sub.to_csv(OUT / "submission.csv")
    pd.DataFrame(p_test, columns=CLASSES, index=test.index).rename_axis("lesion_id").to_csv(OUT / "test_probabilities.csv")
    print(f"yazıldı: {(OUT / 'submission.csv').relative_to(ROOT)}")

    counts = np.bincount(y, minlength=len(CLASSES))
    display(pd.DataFrame({"train gerçek %": counts / len(y) * 100,
                          "test tahmin %": sub.sum().values / len(sub) * 100}, index=CLASSES).round(1))

son eğitim (tüm train):   0%|          | 0/3270 [00:00<?, ?adım/s]

gradient checkpointing: açık
epoch  1/10 | loss 2.085 | 3.4 dk
epoch  2/10 | loss 1.737 | 3.8 dk
epoch  3/10 | loss 1.373 | 4.1 dk
epoch  4/10 | loss 1.048 | 4.1 dk
epoch  5/10 | loss 0.805 | 4.1 dk
epoch  6/10 | loss 0.581 | 3.7 dk
epoch  7/10 | loss 0.386 | 3.6 dk
epoch  8/10 | loss 0.289 | 3.6 dk
epoch  9/10 | loss 0.233 | 3.6 dk
epoch 10/10 | loss 0.183 | 3.6 dk
yazıldı: outputs\finetune\submission.csv


,train gerçek %,test tahmin %
AKIEC,5.8,8.8
BCC,48.1,25.5
BEN_OTH,0.8,0.2
BKL,10.4,19.0
DF,1.0,1.7
INF,1.0,1.9
MAL_OTH,0.2,0.0
MEL,8.6,6.5
NV,14.2,11.5
SCCKA,9.0,24.0


## Notlar

- Sonuçları `history.md`'ye yeni bir deney olarak ekle (CV tablosu + varsa leaderboard skoru).
- Gönderilecek dosya: `outputs/finetune/submission.csv`. 01'in submission'ı `outputs/baseline/` altında duruyor.
- Bu notebook'un fold 0 sonucu pilot çalıştırmadan geliyor (aynı ayarlar, aynı fold).